04_integrate_cellchat.R — 对应论文 Figure 5（CellChat crosstalk）+ 整合数据
论文 Methods：
  - Seurat v3 标准整合 (FindIntegrationAnchors + IntegrateData，ref 46 = Stuart 2019)
  - CellChat 用默认参数，computeCommunProb → aggregateNet
  - 四个样本的 integrated dataset 也用于 pySCENIC (notebook 05)
运行：conda run -n epn2_r Rscript projectmd/04_integrate_cellchat.R

In [ ]:
setwd('/home/zlcmc/wslproject')
source('workspace_paths.R')

In [ ]:
suppressPackageStartupMessages({
    library(Seurat)
    library(CellChat)
    library(dplyr)
    library(ggplot2)
    library(patchwork)
})
options(Seurat.object.assay.version = 'v3')

In [ ]:
SAMPLES <- c('GTE001','GTE002','GTE009','GTE012')

In [ ]:
# ---- 1) Seurat v3 CCA 整合 ----
integ_cache <- output_path('fig5/integrated_seurat.rds')
if (file.exists(integ_cache)) {
    cat('载入缓存:', integ_cache, '\n')
    integ <- readRDS(integ_cache)
} else {
    obj_list <- lapply(SAMPLES, function(s) {
        o <- readRDS(output_path(paste0('fig1/', s, '_seurat.rds')))
        o$sample <- s
        o <- NormalizeData(o, verbose = FALSE)
        o <- FindVariableFeatures(o, nfeatures = 2000, verbose = FALSE)
        o
    })
    names(obj_list) <- SAMPLES
    features <- SelectIntegrationFeatures(obj_list, nfeatures = 2000)
    anchors  <- FindIntegrationAnchors(obj_list, anchor.features = features,
                                       dims = 1:30)
    integ    <- IntegrateData(anchors, dims = 1:30)
    DefaultAssay(integ) <- 'integrated'
    integ <- ScaleData(integ, verbose = FALSE)
    integ <- RunPCA(integ, npcs = 50, verbose = FALSE)
    integ <- RunTSNE(integ, dims = 1:30, seed.use = 42)
    integ <- FindNeighbors(integ, dims = 1:30, verbose = FALSE)
    integ <- FindClusters(integ, resolution = 0.5, verbose = FALSE)
    dir.create(output_path('fig5'), showWarnings = FALSE, recursive = TRUE)
    saveRDS(integ, integ_cache)
}
cat(sprintf('整合后共 %d 细胞\n', ncol(integ)))

In [ ]:
# ---- 2) 导出 log-normalized 矩阵给 pySCENIC (notebook 05) ----
scenic_dir <- output_path('fig6/scenic_input')
dir.create(scenic_dir, showWarnings = FALSE, recursive = TRUE)
logmat <- GetAssayData(integ, assay = 'RNA', layer = 'data')
saveRDS(list(expr = logmat, meta = integ@meta.data),
        file.path(scenic_dir, 'integrated_logmat.rds'))
cat('已存 pySCENIC 输入:', file.path(scenic_dir, 'integrated_logmat.rds'), '\n')

In [ ]:
# ---- 3) CellChat ----
# Figure 5A: 每个样本各 cell_type 细胞数
meta <- integ@meta.data
if ('Brief_cluster' %in% colnames(meta)) {
    group_col <- 'Brief_cluster'
} else if ('cell_type' %in% colnames(meta)) {
    group_col <- 'cell_type'
} else {
    stop('找不到 Brief_cluster / cell_type 列')
}
cat('分组使用:', group_col, '\n')

In [ ]:
cnt <- meta %>% count(sample, .data[[group_col]])
p_cnt <- ggplot(cnt, aes_string(x='sample', y='n', fill=group_col)) +
         geom_bar(stat='identity') +
         theme_classic() + ylab('# cells') +
         ggtitle('Figure 5A — cell numbers per sample')
ggsave(output_path('fig5/Fig5A_cell_numbers.pdf'), p_cnt, width=7, height=5)

In [ ]:
# CellChat 主流程（默认参数）
cc_cache <- output_path('fig5/cellchat_integrated.rds')
if (file.exists(cc_cache)) {
    cellchat <- readRDS(cc_cache)
} else {
    data_use   <- GetAssayData(integ, assay='RNA', layer='data')
    labels     <- integ@meta.data[[group_col]]
    meta_cc    <- data.frame(group = labels, row.names = colnames(data_use))
    cellchat   <- createCellChat(object = data_use, meta = meta_cc, group.by = 'group')
    cellchat@DB <- CellChatDB.human
    cellchat <- subsetData(cellchat)
    future::plan('multisession', workers = 2)
    cellchat <- identifyOverExpressedGenes(cellchat)
    cellchat <- identifyOverExpressedInteractions(cellchat)
    cellchat <- computeCommunProb(cellchat, raw.use = TRUE)
    cellchat <- filterCommunication(cellchat, min.cells = 10)
    cellchat <- computeCommunProbPathway(cellchat)
    cellchat <- aggregateNet(cellchat)
    saveRDS(cellchat, cc_cache)
}

In [ ]:
# Figure 5B: crosstalk net (circle plot)
groupSize <- as.numeric(table(cellchat@idents))
pdf(output_path('fig5/Fig5B_crosstalk_net.pdf'), width=9, height=9)
par(mfrow = c(1, 2), xpd = TRUE)
netVisual_circle(cellchat@net$count, vertex.weight = groupSize,
                 weight.scale = TRUE, label.edge = FALSE,
                 title.name = '# interactions')
netVisual_circle(cellchat@net$weight, vertex.weight = groupSize,
                 weight.scale = TRUE, label.edge = FALSE,
                 title.name = 'Interaction strength')
dev.off()

In [ ]:
# Supp Fig 8A: cell-type interaction strength heatmap
pdf(output_path('fig5/SuppFig8A_interaction_heatmap.pdf'), width=8, height=6)
print(netVisual_heatmap(cellchat, measure = 'weight'))
dev.off()

In [ ]:
cat('\n✓ Figure 5 + Supp Fig 8 CellChat 完成\n')